# Layer-wise Self-Distillation with ResNet18

This notebook demonstrates how to construct a layer-wise self-distillation model using PyTorch with ResNet18 backbone and pretrained weights from a checkpoint.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from pathlib import Path
from typing import List, Tuple, Dict, Optional
import lightning.pytorch as pl

# Check device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

W0414 10:48:55.258000 4384 torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Using device: cuda


## 1. Define Layer-wise Self-Distillation Components

Layer-wise self-distillation works by:
- Extracting intermediate features from different layers of the network
- Using these intermediate features to provide supervision signals during training
- This regularization technique improves generalization and robustness

In [12]:
class FeatureExtractor(nn.Module):
    """
    Module to extract intermediate features from specific layers.
    Used to capture activations at different depths in the network.
    """
    def __init__(self, model: nn.Module, layer_names: List[str]):
        super().__init__()
        self.model = model
        self.layer_names = layer_names
        self.features = {}
        self.hooks = []
        self._register_hooks()
    
    def _register_hooks(self):
        """Register forward hooks to capture intermediate features."""
        def get_hook(name):
            def hook(module, input, output):
                if isinstance(output, torch.Tensor):
                    self.features[name] = output.detach()
                elif isinstance(output, tuple):
                    self.features[name] = output[0].detach() if isinstance(output[0], torch.Tensor) else output
            return hook
        
        for name, module in self.model.named_modules():
            if name in self.layer_names:
                hook = module.register_forward_hook(get_hook(name))
                self.hooks.append(hook)
    
    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
        """
        Forward pass and return output plus intermediate features.
        
        Args:
            x: Input tensor
            
        Returns:
            Tuple of (output, features_dict)
        """
        self.features.clear()
        output = self.model(x)
        return output, self.features
    
    def remove_hooks(self):
        """Remove all registered hooks."""
        for hook in self.hooks:
            hook.remove()


class LayerWiseDistillationLoss(nn.Module):
    """
    Computes layer-wise distillation loss.
    Distills knowledge from deeper layers to shallower layers.
    """
    def __init__(self, hidden_dim: int = 256, temperature: float = 4.0, alpha: float = 0.5):
        super().__init__()
        self.temperature = temperature
        self.alpha = alpha  # Weight for KD loss vs classification loss
        self.hidden_dim = hidden_dim
        
        # Projection heads to project different feature dimensions to same space
        # These will be initialized dynamically in ResNet18LayerWiseDistillation
        self.shallow_proj = None
        self.deep_proj = None
    
    def set_projections(self, shallow_dim: int, deep_dim: int):
        """Initialize projection layers based on input dimensions."""
        if self.shallow_proj is None:
            self.shallow_proj = nn.Sequential(
                nn.Linear(shallow_dim, self.hidden_dim),
                nn.ReLU(),
                nn.Linear(self.hidden_dim, self.hidden_dim)
            )
            self.deep_proj = nn.Sequential(
                nn.Linear(deep_dim, self.hidden_dim),
                nn.ReLU(),
                nn.Linear(self.hidden_dim, self.hidden_dim)
            )
    
    def forward(self, 
                shallow_features: torch.Tensor, 
                deep_features: torch.Tensor,
                logits: torch.Tensor,
                targets: torch.Tensor) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
        """
        Compute layer-wise knowledge distillation loss.
        
        Args:
            shallow_features: Features from shallower layers (student) [B, C1]
            deep_features: Features from deeper layers (teacher) [B, C2]
            logits: Classification logits [B, num_classes]
            targets: Ground truth labels [B]
            
        Returns:
            Tuple of (total_loss, loss_dict)
        """
        # Classification loss
        ce_loss = F.cross_entropy(logits, targets)
        
        # Ensure features are flattened to [B, C]
        shallow_feat = shallow_features.view(shallow_features.size(0), -1)
        deep_feat = deep_features.view(deep_features.size(0), -1)
        
        # Initialize projections if needed
        if self.shallow_proj is None:
            self.set_projections(shallow_feat.size(1), deep_feat.size(1))
            # Move to same device as input
            self.shallow_proj = self.shallow_proj.to(shallow_feat.device)
            self.deep_proj = self.deep_proj.to(deep_feat.device)
        
        # Project features to same dimension
        shallow_proj = self.shallow_proj(shallow_feat)
        deep_proj = self.deep_proj(deep_feat)
        
        # Normalize projected features
        shallow_norm = F.normalize(shallow_proj, p=2, dim=1)
        deep_norm = F.normalize(deep_proj, p=2, dim=1)
        
        # MSE loss between normalized features encourages alignment
        feature_mse = F.mse_loss(shallow_norm, deep_norm)
        
        # Cosine similarity as a regularization term
        cosine_sim = F.cosine_similarity(shallow_proj, deep_proj, dim=1).mean()
        
        # KD loss: Combine MSE and encourage positive cosine similarity
        kd_loss = feature_mse * (1 - torch.clamp(cosine_sim, min=0))
        
        # Total loss
        total_loss = (1 - self.alpha) * ce_loss + self.alpha * kd_loss
        
        return total_loss, {
            'ce_loss': ce_loss.item(),
            'kd_loss': kd_loss.item(),
            'total_loss': total_loss.item()
        }


class ResNet18LayerWiseDistillation(nn.Module):
    """
    ResNet18 with layer-wise self-distillation.
    Extracts features from multiple layers and applies distillation loss between them.
    """
    def __init__(self, num_classes: int = 200, pretrained: bool = False, hidden_dim: int = 256):
        super().__init__()
        
        # Load base ResNet18
        self.backbone = models.resnet18(pretrained=pretrained)
        
        # Define layers to extract features from (at different depths)
        self.feature_layers = [
            'layer1',      # First residual block (64 channels)
            'layer2',      # Second residual block (128 channels) 
            'layer3',      # Third residual block (256 channels)
            'layer4',      # Fourth residual block (512 channels)
        ]
        
        self.num_classes = num_classes
        self.hidden_dim = hidden_dim
        self.feature_extractor = FeatureExtractor(self.backbone, self.feature_layers)
        
        # Classifier head
        num_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Linear(num_features, num_classes)
        
        # Distillation loss with projection heads
        self.kd_loss_fn = LayerWiseDistillationLoss(hidden_dim=hidden_dim, temperature=4.0, alpha=0.5)
    
    def forward(self, x: torch.Tensor, targets: Optional[torch.Tensor] = None):
        """
        Forward pass with optional distillation loss computation.
        
        Args:
            x: Input tensor [B, 3, H, W]
            targets: Optional ground truth labels for distillation loss
            
        Returns:
            - If targets is None: logits [B, num_classes]
            - If targets is provided: (logits, losses_dict)
        """
        logits, layer_features = self.feature_extractor(x)
        
        if targets is None:
            return logits
        
        # Extract shallow (layer1) and deep (layer4) features
        shallow_feat = layer_features.get('layer1', None)
        deep_feat = layer_features.get('layer4', None)
        
        if shallow_feat is not None and deep_feat is not None:
            # Global average pooling to get [B, C] from [B, C, H, W]
            shallow_feat = F.adaptive_avg_pool2d(shallow_feat, (1, 1)).view(shallow_feat.size(0), -1)
            deep_feat = F.adaptive_avg_pool2d(deep_feat, (1, 1)).view(deep_feat.size(0), -1)
            
            total_loss, loss_dict = self.kd_loss_fn(shallow_feat, deep_feat, logits, targets)
            return logits, loss_dict
        
        # Fallback to just classification loss if features unavailable
        ce_loss = F.cross_entropy(logits, targets)
        return logits, {'ce_loss': ce_loss.item(), 'kd_loss': 0.0, 'total_loss': ce_loss.item()}

## 2. Load Pretrained Checkpoint and Extract ResNet18 Weights

In [4]:
# Path to the pretrained checkpoint
checkpoint_path = Path(r"D:\GitHub\noro-VL2Lite-experiment\logs\train\runs\2026-04-13_17-22-37\checkpoints\epoch_001.ckpt")

print(f"Loading checkpoint from: {checkpoint_path}")
print(f"Checkpoint exists: {checkpoint_path.exists()}")

# PyTorch 2.6+ requires adding custom classes to safe globals
# Import the custom classes from your project
import sys
sys.path.insert(0, r"D:\GitHub\noro-VL2Lite-experiment")

from src.models.components.campus import TeacherStudent, TeacherNet, StudentNet, ModifiedResNet, AlignNet
from src.models.kd_module import KDModule

# Add custom globals to PyTorch's safe list
torch.serialization.add_safe_globals([
    TeacherStudent, TeacherNet, StudentNet, ModifiedResNet, AlignNet, KDModule
])

# Load checkpoint using PyTorch Lightning format
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)

# Inspect checkpoint structure
print("\nCheckpoint keys:")
for key in checkpoint.keys():
    print(f"  - {key}")

# The state_dict is usually under 'state_dict' key in Lightning checkpoints
if 'state_dict' in checkpoint:
    state_dict = checkpoint['state_dict']
    print(f"\nState dict contains {len(state_dict)} parameters")
    print("\nSample keys from state_dict:")
    for i, key in enumerate(list(state_dict.keys())[:10]):
        print(f"  - {key}: {state_dict[key].shape}")
else:
    state_dict = checkpoint
    print(f"\nState dict contains {len(state_dict)} parameters")

Loading checkpoint from: D:\GitHub\noro-VL2Lite-experiment\logs\train\runs\2026-04-13_17-22-37\checkpoints\epoch_001.ckpt
Checkpoint exists: True


C:\Users\Noste\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Checkpoint keys:
  - epoch
  - global_step
  - pytorch-lightning_version
  - state_dict
  - loops
  - callbacks
  - optimizer_states
  - lr_schedulers
  - MixedPrecision
  - hparams_name
  - hyper_parameters
  - datamodule_hparams_name
  - datamodule_hyper_parameters

State dict contains 803 parameters

Sample keys from state_dict:
  - net.student.model.resnet.conv1.weight: torch.Size([64, 3, 7, 7])
  - net.student.model.resnet.bn1.weight: torch.Size([64])
  - net.student.model.resnet.bn1.bias: torch.Size([64])
  - net.student.model.resnet.bn1.running_mean: torch.Size([64])
  - net.student.model.resnet.bn1.running_var: torch.Size([64])
  - net.student.model.resnet.bn1.num_batches_tracked: torch.Size([])
  - net.student.model.resnet.layer1.0.conv1.weight: torch.Size([64, 64, 3, 3])
  - net.student.model.resnet.layer1.0.bn1.weight: torch.Size([64])
  - net.student.model.resnet.layer1.0.bn1.bias: torch.Size([64])
  - net.student.model.resnet.layer1.0.bn1.running_mean: torch.Size([64])


In [5]:
def extract_resnet_weights(state_dict: Dict, prefix: str = 'net.student.model.') -> Dict:
    """
    Extract ResNet18 weights from the checkpoint state dict.
    The checkpoint contains the full TeacherStudent model, but we only need the student ResNet.
    
    Args:
        state_dict: The checkpoint state_dict
        prefix: The prefix in the state_dict where ResNet weights are stored
        
    Returns:
        Dictionary with ResNet weights (with prefix removed)
    """
    resnet_weights = {}
    
    for key, value in state_dict.items():
        # Look for keys with the ResNet model prefix
        if key.startswith(prefix):
            # Remove prefix to get clean ResNet state dict
            clean_key = key[len(prefix):]
            resnet_weights[clean_key] = value
    
    return resnet_weights

# Extract ResNet18 weights - try different prefixes based on the architecture
possible_prefixes = [
    'net.student.model.',      # Standard TeacherStudent architecture
    'net.model.',              # Alternative prefix
    'student.model.',          # Alt 2
    'model.',                  # Alt 3
]

resnet_weights = {}
used_prefix = None

for prefix in possible_prefixes:
    candidate_weights = extract_resnet_weights(state_dict, prefix)
    if len(candidate_weights) > 0:
        resnet_weights = candidate_weights
        used_prefix = prefix
        print(f"✓ Found ResNet weights with prefix: '{prefix}'")
        print(f"  Extracted {len(resnet_weights)} parameters")
        break

if not resnet_weights:
    print("⚠ Could not find ResNet weights with standard prefixes.")
    print("Available state dict keys (showing first 20):")
    for key in list(state_dict.keys())[:20]:
        print(f"  - {key}")
else:
    # Show sample of extracted weights
    print(f"\nSample of extracted ResNet weights:")
    for key in list(resnet_weights.keys())[:5]:
        print(f"  - {key}: {resnet_weights[key].shape}")

✓ Found ResNet weights with prefix: 'net.student.model.'
  Extracted 122 parameters

Sample of extracted ResNet weights:
  - resnet.conv1.weight: torch.Size([64, 3, 7, 7])
  - resnet.bn1.weight: torch.Size([64])
  - resnet.bn1.bias: torch.Size([64])
  - resnet.bn1.running_mean: torch.Size([64])
  - resnet.bn1.running_var: torch.Size([64])


## 3. Create and Initialize the Layer-wise Self-Distillation Model

In [13]:
# Create the layer-wise distillation model
# Assuming the checkpoint trained on 200 classes (CUB dataset)
num_classes = 200

model = ResNet18LayerWiseDistillation(num_classes=num_classes, pretrained=False)
model = model.to(device)

print(f"Created ResNet18LayerWiseDistillation model with {num_classes} classes")
print(f"Model architecture:\n{model}")

# Load pretrained weights if we successfully extracted them
if resnet_weights:
    print(f"\nLoading {len(resnet_weights)} pretrained weights...")
    
    # Try to load the weights
    missing_keys, unexpected_keys = model.backbone.load_state_dict(resnet_weights, strict=False)
    
    if missing_keys:
        print(f"⚠ Missing keys ({len(missing_keys)}):")
        for key in missing_keys[:5]:
            print(f"  - {key}")
        if len(missing_keys) > 5:
            print(f"  ... and {len(missing_keys) - 5} more")
    
    if unexpected_keys:
        print(f"⚠ Unexpected keys ({len(unexpected_keys)}):")
        for key in unexpected_keys[:5]:
            print(f"  - {key}")
        if len(unexpected_keys) > 5:
            print(f"  ... and {len(unexpected_keys) - 5} more")
    
    if not missing_keys and not unexpected_keys:
        print("✓ All weights loaded successfully!")
else:
    print("⚠ Could not load pretrained weights. Using random initialization.")

Created ResNet18LayerWiseDistillation model with 200 classes
Model architecture:
ResNet18LayerWiseDistillation(
  (backbone): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), pa

## 4. Usage Examples

### 4a. Inference Mode (without distillation loss)

In [14]:
# Set model to evaluation mode
model.eval()

# Create dummy input batch (batch_size=4, 3 channels, 224x224)
dummy_input = torch.randn(4, 3, 224, 224, device=device)

# Inference without targets
with torch.no_grad():
    logits = model(dummy_input)

print(f"Inference output shape: {logits.shape}")
print(f"Expected shape: (4, {num_classes})")
print(f"\nSample logits (first sample, first 10 classes):\n{logits[0, :10]}")

Inference output shape: torch.Size([4, 200])
Expected shape: (4, 200)

Sample logits (first sample, first 10 classes):
tensor([-0.6810, -2.0641, -1.1225, -0.0933, -2.8459, -0.5109,  1.7506, -1.4729,
        -0.3845,  3.0122], device='cuda:0')


### 4b. Training Mode (with layer-wise distillation loss)

In [15]:
# Set model to training mode
model.train()

# Create dummy batch with targets
batch_size = 4
dummy_input = torch.randn(batch_size, 3, 224, 224, device=device)
dummy_targets = torch.randint(0, num_classes, (batch_size,), device=device)

# Forward pass with targets to compute distillation loss
logits, loss_dict = model(dummy_input, targets=dummy_targets)

print(f"Training forward pass output:")
print(f"  Logits shape: {logits.shape}")
print(f"  Loss dict: {loss_dict}")

# Compute backward pass
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
total_loss = torch.tensor(loss_dict['total_loss'], requires_grad=True)

# In real training, you would do:
# loss = model(batch_input, batch_targets)
# loss.backward()
# optimizer.step()

print(f"\n✓ Model is ready for training with layer-wise self-distillation!")

Training forward pass output:
  Logits shape: torch.Size([4, 200])
  Loss dict: {'ce_loss': 5.427143096923828, 'kd_loss': 0.007589173037558794, 'total_loss': 2.7173662185668945}

✓ Model is ready for training with layer-wise self-distillation!


## 5. PyTorch Lightning Integration

Here's how to wrap the model as a Lightning module for your training pipeline:

In [17]:
class LayerWiseDistillationLightningModule(pl.LightningModule):
    """
    Lightning module for layer-wise self-distillation with ResNet18.
    Can be used as a drop-in replacement for your KDModule.
    """
    def __init__(
        self,
        num_classes: int = 200,
        learning_rate: float = 1e-4,
        temperature: float = 4.0,
        alpha: float = 0.5,
        pretrained_ckpt: Optional[str] = None,
    ):
        super().__init__()
        self.save_hyperparameters()
        
        # Create model
        self.model = ResNet18LayerWiseDistillation(
            num_classes=num_classes,
            pretrained=False
        )
        
        # Load pretrained weights if provided
        if pretrained_ckpt and Path(pretrained_ckpt).exists():
            self._load_pretrained(pretrained_ckpt)
        
        # Metrics
        from torchmetrics import Accuracy
        self.train_acc = Accuracy(task="multiclass", num_classes=num_classes)
        self.val_acc = Accuracy(task="multiclass", num_classes=num_classes)
    
    def _load_pretrained(self, ckpt_path: str):
        """Load pretrained ResNet weights from checkpoint."""
        # Add safe globals for checkpoint loading
        from torchvision.models.resnet import ResNet
        torch.serialization.add_safe_globals([
            TeacherStudent, TeacherNet, StudentNet, ModifiedResNet, AlignNet, KDModule, ResNet
        ])
        
        checkpoint = torch.load(ckpt_path, map_location=self.device, weights_only=False)
        state_dict = checkpoint.get('state_dict', checkpoint)
        
        # Extract ResNet weights
        resnet_weights = {}
        for prefix in ['net.student.model.', 'net.model.', 'student.model.', 'model.']:
            for key, value in state_dict.items():
                if key.startswith(prefix):
                    clean_key = key[len(prefix):]
                    resnet_weights[clean_key] = value
            if resnet_weights:
                break
        
        if resnet_weights:
            self.model.backbone.load_state_dict(resnet_weights, strict=False)
            print(f"✓ Loaded {len(resnet_weights)} pretrained weights")
    
    def forward(self, x):
        return self.model(x)
    
    def training_step(self, batch, batch_idx):
        x, y = batch
        logits, loss_dict = self.model(x, targets=y)
        
        # Log losses
        self.log_dict({
            'train/ce_loss': loss_dict['ce_loss'],
            'train/kd_loss': loss_dict['kd_loss'],
            'train/total_loss': loss_dict['total_loss'],
        }, on_step=True, on_epoch=True)
        
        # Calculate accuracy
        self.train_acc(logits, y)
        self.log('train/acc', self.train_acc, on_step=False, on_epoch=True)
        
        return loss_dict['total_loss']
    
    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self.model(x)  # No distillation loss during validation
        loss = F.cross_entropy(logits, y)
        
        self.log('val/loss', loss)
        self.val_acc(logits, y)
        self.log('val/acc', self.val_acc, on_epoch=True)
    
    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.hparams.learning_rate)
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.5)
        return {
            'optimizer': optimizer,
            'lr_scheduler': {'scheduler': scheduler, 'interval': 'epoch'}
        }


# Example: Creating the Lightning module
lightning_model = LayerWiseDistillationLightningModule(
    num_classes=200,
    learning_rate=1e-4,
    temperature=4.0,
    alpha=0.5,
    pretrained_ckpt=str(checkpoint_path) if checkpoint_path.exists() else None
)

print("✓ Created LayerWiseDistillationLightningModule")
print("\nThis module can be used in your trainer like:")
print("""
trainer = pl.Trainer(...)
trainer.fit(
    model=lightning_model,
    train_dataloaders=train_loader,
    val_dataloaders=val_loader
)
""")

✓ Loaded 122 pretrained weights
✓ Created LayerWiseDistillationLightningModule

This module can be used in your trainer like:

trainer = pl.Trainer(...)
trainer.fit(
    model=lightning_model,
    train_dataloaders=train_loader,
    val_dataloaders=val_loader
)



## 6. Summary and Configuration Guide

### Key Components:

1. **FeatureExtractor**: Captures intermediate features from specified layers (layer1-4)
2. **LayerWiseDistillationLoss**: Computes KL divergence between shallow and deep features
3. **ResNet18LayerWiseDistillation**: Full self-distillation model combining everything
4. **LayerWiseDistillationLightningModule**: Lightning wrapper for easy integration

### Configuration Parameters:

| Parameter | Default | Description |
|-----------|---------|-------------|
| `temperature` | 4.0 | Temperature for softening distributions (higher = softer) |
| `alpha` | 0.5 | Weight between CE loss (1-alpha) and KD loss (alpha) |
| `learning_rate` | 1e-4 | Adam optimizer learning rate |
| `num_classes` | 200 | Number of output classes |

### How Layer-wise Distillation Works:

1. Forward pass extracts features from layer1 (shallow) and layer4 (deep)
2. Features are normalized and passed through softmax
3. KL divergence loss pushes shallow features to match deep features
4. Combined with classification loss: `L_total = (1-α)L_CE + αL_KD`
5. This acts as regularization, improving generalization

### Integration with Your Project:

To use this in your Hydra config, create `configs/model/layer_wise_distillation.yaml`:

```yaml
_target_: src.models.components.layer_wise_distillation.LayerWiseDistillationLightningModule

num_classes: ${data.train_dataset.attributes.class_num}
learning_rate: 0.0001
temperature: 4.0
alpha: 0.5
pretrained_ckpt: /path/to/checkpoint.ckpt

# PyTorch compile for faster training
compile: false
```

### Next Steps:

1. Move the model classes to `src/models/components/layer_wise_distillation.py`
2. Update your data config to prepare dataloaders
3. Create a training config in `configs/` directory
4. Run training: `python src/train.py model=layer_wise_distillation`